# MusicGen Dreamboothing on Kaggle

This notebook is optimized for running on Kaggle with T4 GPUs (multi-GPU supported).

In [ ]:
%%capture
# 1. Install dependencies
!git clone https://github.com/Phan-Trung-Thuan/musicgen-dreamboothing
%cd musicgen-dreamboothing
!pip install -r requirements-kaggle.txt
!pip install -e .

In [ ]:
# 2. Prepare Dataset
# Update these paths to your Kaggle input
SOURCE_DIR = "/kaggle/input/datasets/phantrungthuan/vietnamese-music-dataset"
CAPTION_FILE = "/kaggle/input/datasets/phantrungthuan/vietnamese-music-dataset/vietnamese-music-captioning.json"
DEST_DIR = "./musicfiles"

!python prepare_kaggle_dataset.py \
    --source_dir "{SOURCE_DIR}" \
    --caption_file "{CAPTION_FILE}" \
    --dest_dir "{DEST_DIR}"

from datasets import load_dataset
dataset = load_dataset("audiofolder", data_dir=DEST_DIR)
for split in dataset.keys():
    print(f"Number of samples used for {split}: {len(dataset[split])}")

In [ ]:
# 3. Run Training
import os
OUTPUT_DIR = "./musicgen-vietnamese-lora"

command = (
    f"accelerate launch --multi_gpu --mixed_precision=fp16 dreambooth_musicgen.py "
    f"    --model_name_or_path facebook/musicgen-small "
    f"    --dataset_name {os.path.abspath(DEST_DIR)} "
    f"    --target_audio_column_name audio "
    f"    --text_column_name text "
    f"    --output_dir {OUTPUT_DIR} "
    f"    --use_lora "
    f"    --do_train "
    f"    --fp16 "
    f"    --num_train_epochs 5 "
    f"    --learning_rate 2e-4 "
    f"    --per_device_train_batch_size 2 "
    f"    --gradient_accumulation_steps 4 "
    f"    --save_steps 50 "
    f"    --logging_steps 10 "
    f"    --train_split_name train "
    f"    --decoder_start_token_id 2047 "
    f"    --pad_token_id 2047"
)

print(f"Running command: {command}")
!{command}

# 4. Inference
Load the fine-tuned LoRA model and generate audio.

In [ ]:
import torch
from transformers import AutoProcessor, MusicgenForConditionalGeneration
import IPython.display as ipd

# 1. Load base model and processor
model_id = "facebook/musicgen-small"
processor = AutoProcessor.from_pretrained(model_id)
model = MusicgenForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16)

# 2. Load LoRA weights
model.load_adapter(OUTPUT_DIR)
model.to("cuda")

prompts = ["Vietnamese folk music with traditional instruments"]
inputs = processor(text=prompts, padding=True, return_tensors="pt").to("cuda")

with torch.no_grad():
    audio_values = model.generate(**inputs, max_new_tokens=512)

sampling_rate = model.config.audio_encoder.sampling_rate
for audio in audio_values:
    ipd.display(ipd.Audio(audio[0].cpu().numpy(), rate=sampling_rate))

# 5. Archive Model
Zip the trained model folder for downloading.

In [ ]:
!zip -r musicgen_lora_model.zip {OUTPUT_DIR}